In [1]:
# 启用自动重载功能，便于开发调试
# 当模块代码发生变化时，自动重新加载，无需重启内核
%load_ext autoreload
%autoreload 2

# ==================== 基础库导入 ====================
import open3d  # 3D点云处理和可视化库，优先导入以避免潜在的导入冲突
import numpy as np  # 数值计算和数组操作库，用于处理分子坐标和属性数据
from rdkit import Chem  # RDKit化学信息学库，提供分子对象操作和化学性质计算功能

# ==================== 分子构象生成 ====================
# from shepherd_score.conformer_generation import embed_conformer_from_smiles
# embed_conformer_from_smiles: 核心函数，从SMILES字符串生成优化的3D分子构象
# 功能：SMILES解析 -> 3D坐标生成 -> 力场优化 -> 返回RDKit分子对象

# ==================== 评估管道系统 ====================
from shepherd_score.evaluations.evaluate import ConfEval, UnconditionalEvalPipeline
from shepherd_score.evaluations.evaluate import ConsistencyEvalPipeline, ConditionalEvalPipeline
# ConfEval: 单分子构象评估类，提供基础的分子性质计算功能
# UnconditionalEvalPipeline: 无条件评估管道，评估生成分子的化学合理性和多样性
# ConsistencyEvalPipeline: 一致性评估管道，评估生成分子间相互作用轮廓的一致性
# ConditionalEvalPipeline: 条件评估管道，评估生成分子与参考分子的相似性匹配度

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
import rdkit
import rdkit.Chem
import rdkit.Chem.AllChem
from rdkit.Geometry import Point3D
from rdkit.Chem import rdDistGeom
from rdkit.Chem import rdMolAlign
from rdkit.ML.Cluster import Butina

In [3]:


def embed_conformer(mol: rdkit.Chem.Mol, attempts: int=50, MMFF_optimize: bool=False):
    """
    Embeds a mol object into a 3D RDKit mol object with ETKDG (and optional MMFF94)

    Args:
        mol -- RDKit Mol object
        attempts -- (int) number of embedding attempts
        MMFF_optimize -- (bool) whether to optimize embedded conformer with MMFF94
        
    Returns:
        mol -- RDKit mol object with 3D coordinates
    """
    # try:
    mol = rdkit.Chem.AddHs(mol)
    rdkit.Chem.AllChem.EmbedMolecule(mol, maxAttempts = attempts)
    if MMFF_optimize:
        rdkit.Chem.AllChem.MMFFOptimizeMolecule(mol)

    mol.GetConformer() # test whether conformer generation succeeded

    # except Exception as e:
    #     return None

    return mol


def embed_conformer_from_smiles(smiles: str, attempts: int = 50, MMFF_optimize: bool = False):
    """
    Embeds a SMILES into a 3D RDKit mol object with ETKDG (and optionally MMFF94)
    
    Args:
        smiles -- SMILES string of molecule
        attempts -- (int) number of embedding attempts
        MMFF_optimize -- (bool) whether to optimize embedded conformer with MMFF94
        
    Returns:
        mol -- RDKit mol object with 3D coordinates
    """
    try:
        mol = rdkit.Chem.MolFromSmiles(smiles)
    except Exception as e:
        print('Error in SMILES string when embedding molecule:', e)
        return None

    mol = embed_conformer(mol, attempts, MMFF_optimize)

    return mol

In [4]:
# 从复杂的SMILES字符串生成分子构象，使用MMFF94力场优化
# 这是一个包含氯原子、羰基和杂环的复杂分子结构
rdkit_mol = embed_conformer_from_smiles('c1Cc2ccc(Cl)cc2C(=O)c1c3cc(N1nnc2cc(C)c(Cl)cc2c1=O)ccc3', MMFF_optimize=True)

# 提取原子序数数组 - 每个原子的原子序数（如C=6, N=7, O=8, Cl=17等）
atoms = np.array([a.GetAtomicNum() for a in rdkit_mol.GetAtoms()])
# 获取原子的三维坐标位置矩阵 (N_atoms × 3)
positions = rdkit_mol.GetConformer().GetPositions()

conf_eval = ConfEval(atoms, positions, solvent='water')

conf_eval.to_pandas()

xyz_block                   46\n\nC     -3.53247021     -1.38821552      1...
mol                          <rdkit.Chem.rdchem.Mol object at 0x7f1704bbe970>
smiles                      Cc1cc2nnn(-c3cccc(C4=CCc5ccc(Cl)cc5C4=O)c3)c(=...
molblock                    \n     RDKit          3D\n\n 46 50  0  0  0  0...
energy                                                             -85.047789
partial_charges             [-0.01934644, -0.0876247, 0.0235882, -0.041822...
solvent                                                                 water
charge                                                                      0
xyz_block_post_opt          46\n\nC           -3.75091169978814       -1.5...
mol_post_opt                 <rdkit.Chem.rdchem.Mol object at 0x7f1704bbea50>
smiles_post_opt             Cc1cc2nnn(-c3cccc(C4=CCc5ccc(Cl)cc5C4=O)c3)c(=...
molblock_post_opt           \n     RDKit          3D\n\n 46 50  0  0  0  0...
energy_post_opt                                                 